### Name: Riya Shyam Huddar (MDS202431) 
### Data Pipeline

In [1]:
import os
import pandas as pd
import numpy as np
import cvxpy as cp
from scipy import sparse
import clarabel
import matplotlib.pyplot as plt
import time

### Data Cleansing and Consolidation

The NIFTY 50 dataset is provided as individual CSV files, with each file corresponding to a single stock.  
To enable portfolio-level analysis and optimization, the data is first cleaned and consolidated into a unified dataset.

The following preprocessing steps are performed:

- All stock-wise CSV files are loaded from the dataset directory.
- Columns containing only missing values are removed to eliminate redundant information.
- A new identifier column, `Symbol`, is added to each record to denote the corresponding stock.
- All individual stock datasets are concatenated into a single consolidated dataframe.

In [3]:
data_dir = r"D:\CMI\IP\archive"

dfs = []

for file in os.listdir(data_dir):
    if file.endswith(".csv"):
        symbol = file.replace(".csv", "")
        file_path = os.path.join(data_dir, file)
        
        df = pd.read_csv(file_path)

        # Drop completely empty columns 
        df = df.dropna(axis=1, how="all")

        df["Symbol"] = symbol
        dfs.append(df)

raw_data = pd.concat(dfs, ignore_index=True)


In [4]:
raw_data.head()

,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble,Company Name,Industry,ISIN Code
0,2007-11-27,ADANIPORTS,EQ,440.00,770.00,1050.00,770.0,959.0,962.90,984.72,27294366.0,2.687719e+15,NaN,9859619.0,0.3612,NaN,NaN,NaN
1,2007-11-28,ADANIPORTS,EQ,962.90,984.00,990.00,874.0,885.0,893.90,941.38,4581338.0,4.312765e+14,NaN,1453278.0,0.3172,NaN,NaN,NaN
2,2007-11-29,ADANIPORTS,EQ,893.90,909.00,914.75,841.0,887.0,884.20,888.09,5124121.0,4.550658e+14,NaN,1069678.0,0.2088,NaN,NaN,NaN
3,2007-11-30,ADANIPORTS,EQ,884.20,890.00,958.00,890.0,929.0,921.55,929.17,4609762.0,4.283257e+14,NaN,1260913.0,0.2735,NaN,NaN,NaN
4,2007-12-03,ADANIPORTS,EQ,921.55,939.75,995.00,922.0,980.0,969.30,965.65,2977470.0,2.875200e+14,NaN,816123.0,0.2741,NaN,NaN,NaN


### Data Type Inspection

After consolidating the dataset, we inspect the data types of each column to ensure correctness and identify any required transformations.

- Price and volume-related fields (e.g., `Open`, `Close`, `VWAP`, `Volume`) are correctly stored as numerical (`float64`) values, making them suitable for return and risk calculations.
- Identifier and categorical fields such as `Date`, `Symbol`, and `Series` are currently stored as object types.
- The `Date` column is represented as a string and will be converted to a datetime format.
- Company metadata columns (`Company Name`, `Industry`, `ISIN Code`) are retained for reference but are not directly used in the optimization process.

In [7]:
raw_data.dtypes

Date                   object
Symbol                 object
Series                 object
Prev Close            float64
Open                  float64
High                  float64
Low                   float64
Last                  float64
Close                 float64
VWAP                  float64
Volume                float64
Turnover              float64
Trades                float64
Deliverable Volume    float64
%Deliverble           float64
Company Name           object
Industry               object
ISIN Code              object
dtype: object

### Date Cleaning and Price Matrix Construction

To enable time-series analysis and portfolio optimization, the dataset is further processed to ensure correct temporal ordering and a suitable data layout.

The following steps are performed:

- The `Date` column is converted from string format to a datetime object to allow time-based operations.
- Rows with invalid or missing dates are removed to maintain temporal consistency.
- The data is sorted by `Symbol` and `Date` to ensure correct chronological ordering for each stock.
- Only the relevant fields (`Date`, `Symbol`, and `Close` price) are retained for return computation.
- In cases where multiple records exist for the same stock on the same date, the last available closing price is retained.
- The data is reshaped into a price matrix using a pivot operation, with dates as rows, stock symbols as columns, and closing prices as values.

In [9]:
raw_data["Date"] = pd.to_datetime(raw_data["Date"], errors="coerce")
raw_data = raw_data.dropna(subset=["Date"])
raw_data["Date"].isnull().sum()

0

In [11]:
raw_data = raw_data.sort_values(["Symbol", "Date"])

In [13]:
price_data = raw_data[["Date", "Symbol", "Close"]].copy()
price_data.head()

,Date,Symbol,Close
0,2007-11-27,ADANIPORTS,962.90
1,2007-11-28,ADANIPORTS,893.90
2,2007-11-29,ADANIPORTS,884.20
3,2007-11-30,ADANIPORTS,921.55
4,2007-12-03,ADANIPORTS,969.30


### Handling Duplicate Records

While consolidating the dataset, we observed that some stocks contain multiple
records for the same trading date. Since portfolio optimization requires a
single closing price per stock per day, such duplicate entries must be resolved
before reshaping the data.

To address this, we group the data by date and stock symbol and retain the last
available closing price for each day. This approach is consistent with the
definition of daily closing prices and ensures that each (Date, Symbol) pair
has a unique value, enabling a valid price matrix construction.


In [15]:
price_data = (
    price_data
    .groupby(["Date", "Symbol"], as_index=False)
    .last()
)

price_data.duplicated(subset=["Date", "Symbol"]).sum()

0

In [17]:
price_matrix = price_data.pivot(
    index="Date",
    columns="Symbol",
    values="Close"
)
price_matrix.head()

Symbol,ADANIPORTS,ASIANPAINT,AXISBANK,BAJAJ-AUTO,BAJAJFINSV,BAJFINANCE,BHARTIARTL,BPCL,BRITANNIA,CIPLA,...,TATAMOTORS,TATASTEEL,TCS,TECHM,TITAN,ULTRACEMCO,UPL,VEDL,WIPRO,ZEEL
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,NaN,381.65,26.70,NaN,NaN,50.75,NaN,399.25,756.90,1457.35,...,216.75,152.45,NaN,NaN,155.70,NaN,NaN,116.35,2724.20,1179.95
2000-01-04,NaN,385.55,26.85,NaN,NaN,48.10,NaN,370.50,754.55,1465.25,...,208.20,150.80,NaN,NaN,147.40,NaN,NaN,114.70,2942.15,1260.65
2000-01-05,NaN,383.00,26.30,NaN,NaN,44.60,NaN,359.95,735.30,1435.05,...,213.25,156.55,NaN,NaN,138.40,NaN,NaN,114.00,2990.10,1176.55
2000-01-06,NaN,377.50,25.95,NaN,NaN,45.25,NaN,380.30,785.65,1355.85,...,222.10,168.25,NaN,NaN,149.50,NaN,NaN,119.30,2932.25,1115.45
2000-01-07,NaN,385.70,24.80,NaN,NaN,42.90,NaN,379.85,848.50,1247.55,...,239.90,171.95,NaN,NaN,146.35,NaN,NaN,116.50,2697.70,1026.25


### Handling Missing Data: Two Analysis Universes

After constructing the price matrix, we observe a substantial amount of missing data arising from differences in stock listing dates. A naive approach of dropping all rows with missing values would lead to a significant loss of historical information.

A diagnostic check shows:

- Original number of trading days: 5,306  
- Days remaining after dropping all missing values: 2,598  
- Percentage of data lost: approximately 51%

Inspection of missing-value counts reveals that a small subset of relatively newer stocks is responsible for a large fraction of the missing observations, while most stocks have long and stable price histories.

To understand the impact of this trade-off, **two alternative analysis universes are constructed and compared**:

**Approach 1: Full Universe**  
All available stocks are retained. Rows containing any missing values are dropped, which restricts the analysis to a shorter common time window shared by all stocks.

**Approach 2: Extended History**  
Stocks with excessive missing data are excluded, and the remaining dataset is filtered to rows without missing values. This reduces the number of stocks slightly but enables a substantially longer historical time span.

Both approaches are carried forward in parallel. Comparing the resulting portfolios allows us to assess how sensitive the optimization results are to the choice of stock universe and data horizon. This comparison provides insight into the trade-off between universe completeness and statistical robustness in portfolio construction.


In [19]:
rows_before = len(price_matrix)
rows_after = len(price_matrix.dropna())

print(f"Original number of days: {rows_before}")
print(f"Days remaining if we drop all NaNs: {rows_after}")
print(f"Percentage of data lost: {100 * (1 - rows_after/rows_before):.2f}%")

Original number of days: 5306
Days remaining if we drop all NaNs: 2598
Percentage of data lost: 51.04%


In [21]:
# stocks with maximum NaN's 
price_matrix.isnull().sum().sort_values(ascending=False).head(10)

Symbol
COALINDIA     2708
NESTLEIND     2500
BAJAJFINSV    2105
BAJAJ-AUTO    2104
ADANIPORTS    1984
POWERGRID     1947
TECHM         1671
JSWSTEEL      1312
NTPC          1218
TCS           1167
dtype: int64

In [159]:
# APPROACH 1: Full Universe 
# We keep every single symbol, which forces the timeline to start 
# only when the "youngest" stock was listed.
price_matrix_full = price_matrix.dropna()

# APPROACH 2: Extended History
# We identify the stocks that are "killing" our history (missing > 1100 days).
# By removing just these few, we "unlock" many more years of data for the rest.
nan_counts = price_matrix.isnull().sum()
killers = nan_counts[nan_counts > 1100].index
price_matrix_extended = price_matrix.drop(columns=killers).dropna()

# --- DIAGNOSTICS ---
print("--- APPROACH 1: FULL UNIVERSE ---")
print(f"Stocks: {price_matrix_full.shape[1]}")
print(f"Days: {len(price_matrix_full)}")
print(f"Timeline: {price_matrix_full.index.min().date()} to {price_matrix_full.index.max().date()}")

print("\n--- APPROACH 2: EXTENDED HISTORY ---")
print(f"Stocks: {price_matrix_extended.shape[1]}")
print(f"Days: {len(price_matrix_extended)}")
print(f"Timeline: {price_matrix_extended.index.min().date()} to {price_matrix_extended.index.max().date()}")

--- APPROACH 1: FULL UNIVERSE ---
Stocks: 50
Days: 2598
Timeline: 2010-11-04 to 2021-04-30

--- APPROACH 2: EXTENDED HISTORY ---
Stocks: 38
Days: 4286
Timeline: 2004-01-23 to 2021-04-30


### Annualized Return and Risk Summary

Annualized mean returns and covariance matrices are computed from daily returns for both analysis universes using standard market conventions.

Across both approaches, high-return stocks are largely consistent, with **BAJFINANCE**, **SHREECEM**, and other growth-oriented stocks appearing among the top performers. Minor differences in rankings arise due to the longer time horizon captured in the extended-history dataset.

The covariance matrices show substantial variation in individual stock risk (diagonal entries) and meaningful co-movement within sectors, particularly among financial stocks. At the same time, lower cross-sector covariances indicate potential diversification benefits.

These results highlight that return alone is insufficient for portfolio selection and motivate the use of mean–variance optimization, where both expected return and risk interactions between assets are explicitly accounted for.
The annualization uses the standard 252-trading-day convention.

In [157]:
def compute_annual_stats(matrix):
    # Calculate daily returns
    daily_ret = matrix.pct_change().dropna()
    
    # Annualize Mean Returns (Mean * 252)
    mu = daily_ret.mean() * 252
    
    # Annualize Covariance (Cov * 252)
    Sigma = daily_ret.cov() * 252
    
    return mu, Sigma

# Stats for Approach 1 (Full 50 stocks, 10 years)
mu_full, Sigma_full = compute_annual_stats(price_matrix_full)

# Stats for Approach 2 (38 stocks, 17 years)
mu_ext, Sigma_ext = compute_annual_stats(price_matrix_extended)

print(f"Stats computed for Approach 1: {len(mu_full)} assets")
print(f"Stats computed for Approach 2: {len(mu_ext)} assets")

Stats computed for Approach 1: 50 assets
Stats computed for Approach 2: 38 assets


In [155]:
import pandas as pd

def print_stats_report(name, mu, sigma):
    print(f"\n{'='*20} {name} {'='*20}")
    
    # Top 5 Best Performing Stocks (Annualized Return)
    print("\nTop 5 Stocks by Annualized Return:")
    top_mu = mu.sort_values(ascending=False).head(5)
    print(top_mu.map(lambda x: f"{x:.2%}"))
    
    # Covariance Matrix Snippet (First 5x5)
    print("\nCovariance Matrix Snippet (First 5 stocks):")
    # We round to 4 decimals
    print(sigma.iloc[:5, :5].round(4))

# Print for both
print_stats_report("APPROACH 1: FULL UNIVERSE", mu_full, Sigma_full)
print_stats_report("APPROACH 2: EXTENDED HISTORY", mu_ext, Sigma_ext)


==================== APPROACH 1: FULL UNIVERSE ====================

Top 5 Stocks by Annualized Return:
Symbol
BAJFINANCE    39.03%
BAJAJFINSV    36.69%
SHREECEM      28.62%
BRITANNIA     25.92%
EICHERMOT     24.44%
dtype: object

Covariance Matrix Snippet (First 5 stocks):
Symbol      ADANIPORTS  ASIANPAINT  AXISBANK  BAJAJ-AUTO  BAJAJFINSV
Symbol                                                              
ADANIPORTS      0.1473      0.0284    0.0553      0.0258      0.0389
ASIANPAINT      0.0284      0.1470    0.0308      0.0245      0.0264
AXISBANK        0.0553      0.0308    0.2003      0.0352      0.0540
BAJAJ-AUTO      0.0258      0.0245    0.0352      0.0691      0.0289
BAJAJFINSV      0.0389      0.0264    0.0540      0.0289      0.1254

==================== APPROACH 2: EXTENDED HISTORY ====================

Top 5 Stocks by Annualized Return:
Symbol
BAJFINANCE    42.46%
SHREECEM      38.24%
TITAN         35.22%
INDUSINDBK    30.25%
EICHERMOT     29.75%
dtype: object

Covari

In [33]:
def find_min_risk(sigma):
    n = sigma.shape[0]
    w = cp.Variable(n)
    
    # Objective: Minimize Variance (w.T @ Sigma @ w)
    objective = cp.Minimize(cp.quad_form(w, sigma.values))
    constraints = [cp.sum(w) == 1, w >= 0]
    
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.CLARABEL)
    
    # Risk is the square root of Variance
    min_vol = np.sqrt(prob.value)
    return min_vol

# Calculate for both
min_risk_full = find_min_risk(Sigma_full)
min_risk_ext = find_min_risk(Sigma_ext)

print(f"--- MINIMUM ACHIEVABLE RISK (GMV) ---")
print(f"Full Universe (10 Years):    {min_risk_full:.2%}")
print(f"Extended Universe (17 Years): {min_risk_ext:.2%}")
print("-" * 38)

if min_risk_full < min_risk_ext:
    print(f"OBSERVATION: The 'Safety Floor' is {min_risk_ext - min_risk_full:.2%} higher in the Extended Universe.")
    print("This is likely due to the inclusion of the 2008 Financial Crisis data.")

--- MINIMUM ACHIEVABLE RISK (GMV) ---
Full Universe (10 Years):    13.43%
Extended Universe (17 Years): 17.00%
--------------------------------------
OBSERVATION: The 'Safety Floor' is 3.57% higher in the Extended Universe.
This is likely due to the inclusion of the 2008 Financial Crisis data.
